# Día 6 · Teoría de decisión y decisiones metodológicas

**Apartados del documento que cubre este día:** 7 (Teoría de decisión) y la sección obligatoria "Decisiones metodológicas".

**Al terminar este día, cada integrante del equipo debe poder explicar sin leer:**
1. Qué decisión se tomó y con qué números.
2. De dónde sale el umbral de 0.20.
3. Por qué se usan los límites del intervalo como escenarios.
4. Qué dicen los tres criterios de decisión.
5. Qué pasaría si la campaña funcionara peor de lo supuesto.
6. Las respuestas a las ocho preguntas de decisiones metodológicas.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

RAIZ = Path.cwd().parent if Path.cwd().name == "clases" else Path.cwd()
poblacion = pd.read_csv(RAIZ / "data" / "processed" / "poblacion_clientes.csv")
muestra = pd.read_csv(RAIZ / "data" / "processed" / "muestra_clientes_n1000.csv")
N = len(poblacion)
n = len(muestra)
print("Clientes en la población (N):", N)
print("Clientes en la muestra (n):", n)

Clientes en la población (N): 10000
Clientes en la muestra (n): 1000


---
## Paso 1 · Qué pide el documento

El apartado 7 pide plantear una **decisión basada en evidencia**. El documento advierte que no basta decir "el servidor A es mejor". Hay que considerar:
- La diferencia observada.
- El intervalo de confianza.
- La incertidumbre.
- El costo.
- Las consecuencias de equivocarse.

Cada uno de estos cinco puntos aparece en un paso de este cuaderno.

---
## Paso 2 · La decisión

Si fuéramos responsables del área de retención de clientes del banco, ¿a quién dirigiríamos una campaña para evitar que se vayan?

| Opción | Descripción |
|---|---|
| A1 | No hacer campaña |
| A2 | Campaña sólo a clientes **inactivos** |
| A3 | Campaña a **todos** los clientes |

La campaña consiste en que un asesor llame al cliente y le ofrezca una bonificación.

---
## Paso 3 · Los supuestos económicos

Los datos no dicen cuánto cuesta una campaña. Se usan tres valores **supuestos** (ilustrativos):

| Símbolo | Qué es | Valor |
|---|---|---|
| $c$ | Costo de contactar a un cliente | 25 € |
| $e$ | Efectividad: de cada 100 clientes que iban a irse, cuántos se quedan gracias a la campaña | 25 % |
| $V$ | Ganancia anual que deja un cliente retenido | 500 € |

**Beneficio esperado por cliente contactado:**

$$B = p \cdot e \cdot V - c$$

Ejemplo con números: se contacta a 100 clientes de un grupo donde abandona el 20 % ($p$ = 0.20).
- De los 100, se iban a ir 20.
- La campaña retiene al 25 % de esos 20: 5 clientes.
- Esos 5 clientes dejan 5 × 500 € = 2,500 €.
- Contactar a los 100 costó 100 × 25 € = 2,500 €.
- Beneficio: 2,500 € − 2,500 € = 0 €.

In [2]:
costo_contacto = 25
efectividad = 0.25
valor_cliente = 500
clientes_contactados = 100
p_ejemplo = 0.20
clientes_que_se_iban = clientes_contactados * p_ejemplo
clientes_retenidos = clientes_que_se_iban * efectividad
ganancia = clientes_retenidos * valor_cliente
costo_total = clientes_contactados * costo_contacto
beneficio_ejemplo = ganancia - costo_total
print("Clientes que se iban a ir:", clientes_que_se_iban)
print("Clientes retenidos:", clientes_retenidos)
print("Ganancia por clientes retenidos (€):", ganancia)
print("Costo de la campaña (€):", costo_total)
print("Beneficio (€):", beneficio_ejemplo)

Clientes que se iban a ir: 20.0
Clientes retenidos: 5.0
Ganancia por clientes retenidos (€): 2500.0
Costo de la campaña (€): 2500
Beneficio (€): 0.0


---
## Paso 4 · El umbral de rentabilidad

El ejemplo dio beneficio 0 justo con $p$ = 0.20. Ese valor es el **umbral**: la proporción de abandono a partir de la cual la campaña gana dinero. Se despeja de $B = 0$:

$$p^* = \frac{c}{e \cdot V} = \frac{25}{0.25 \times 500} = 0.20$$

- Si en un grupo abandonan **más** del 20 %, la campaña gana dinero.
- Si abandonan **menos** del 20 %, la campaña pierde dinero.

In [3]:
umbral = costo_contacto / (efectividad * valor_cliente)
print("Umbral de rentabilidad (p*):", umbral)

Umbral de rentabilidad (p*): 0.2


---
## Paso 5 · Los intervalos de cada grupo

Para decidir hace falta la proporción de abandono **de cada grupo** con su intervalo. Se calcula igual que P1 (Día 3), una vez para cada grupo:

In [4]:
fcpf = np.sqrt((N - n) / (N - 1))
z_critico = stats.norm.ppf(0.975)

abandono_inactivos = muestra.loc[muestra["miembro_activo"] == 0, "abandono"]
n_inactivos = len(abandono_inactivos)
p_inactivos = abandono_inactivos.mean()
error_inactivos = np.sqrt(p_inactivos * (1 - p_inactivos) / n_inactivos) * fcpf
inferior_inactivos = p_inactivos - z_critico * error_inactivos
superior_inactivos = p_inactivos + z_critico * error_inactivos
print("Inactivos: n =", n_inactivos, "| p gorro =", round(p_inactivos, 4))
print("Inactivos: intervalo al 95 % =", (float(round(inferior_inactivos, 4)), float(round(superior_inactivos, 4))))

abandono_activos = muestra.loc[muestra["miembro_activo"] == 1, "abandono"]
n_activos = len(abandono_activos)
p_activos = abandono_activos.mean()
error_activos = np.sqrt(p_activos * (1 - p_activos) / n_activos) * fcpf
inferior_activos = p_activos - z_critico * error_activos
superior_activos = p_activos + z_critico * error_activos
print("Activos: n =", n_activos, "| p gorro =", round(p_activos, 4))
print("Activos: intervalo al 95 % =", (float(round(inferior_activos, 4)), float(round(superior_activos, 4))))

Inactivos: n = 463 | p gorro = 0.2657
Inactivos: intervalo al 95 % = (0.2275, 0.3038)
Activos: n = 537 | p gorro = 0.1546
Activos: intervalo al 95 % = (0.1256, 0.1836)


**Comparación con el umbral de 0.20:**
- Inactivos: **todo** el intervalo (0.228 a 0.304) está **arriba** de 0.20. La campaña gana dinero aun con el valor más bajo posible.
- Activos: **todo** el intervalo (0.126 a 0.184) está **abajo** de 0.20. La campaña pierde dinero aun con el valor más alto posible.

Esta es la razón principal de la decisión: no se compara sólo la estimación puntual con el umbral, sino **todo el intervalo**. Así se toma en cuenta la incertidumbre.

---
## Paso 6 · Los escenarios

Como no se conoce la proporción exacta de cada grupo, cada opción se evalúa en tres escenarios:

| Escenario | Proporción que se usa |
|---|---|
| Pesimista | Límite inferior del intervalo (el valor más bajo posible) |
| Puntual | La estimación puntual |
| Optimista | Límite superior del intervalo (el valor más alto posible) |

Para pasar del beneficio por cliente al beneficio total se estima cuántos clientes de cada grupo hay en la población:

In [5]:
clientes_inactivos_poblacion = N * n_inactivos / n
clientes_activos_poblacion = N * n_activos / n
print("Clientes inactivos estimados en la población:", clientes_inactivos_poblacion)
print("Clientes activos estimados en la población:", clientes_activos_poblacion)

Clientes inactivos estimados en la población: 4630.0
Clientes activos estimados en la población: 5370.0


In [6]:
beneficio_inactivos_pesimista = (inferior_inactivos * efectividad * valor_cliente - costo_contacto) * clientes_inactivos_poblacion
beneficio_inactivos_puntual = (p_inactivos * efectividad * valor_cliente - costo_contacto) * clientes_inactivos_poblacion
beneficio_inactivos_optimista = (superior_inactivos * efectividad * valor_cliente - costo_contacto) * clientes_inactivos_poblacion
beneficio_activos_pesimista = (inferior_activos * efectividad * valor_cliente - costo_contacto) * clientes_activos_poblacion
beneficio_activos_puntual = (p_activos * efectividad * valor_cliente - costo_contacto) * clientes_activos_poblacion
beneficio_activos_optimista = (superior_activos * efectividad * valor_cliente - costo_contacto) * clientes_activos_poblacion
print("Campaña a inactivos | pesimista:", round(beneficio_inactivos_pesimista), "| puntual:", round(beneficio_inactivos_puntual), "| optimista:", round(beneficio_inactivos_optimista))
print("Campaña a activos   | pesimista:", round(beneficio_activos_pesimista), "| puntual:", round(beneficio_activos_puntual), "| optimista:", round(beneficio_activos_optimista))

Campaña a inactivos | pesimista: 15910 | puntual: 38000 | optimista: 60090
Campaña a activos   | pesimista: -49971 | puntual: -30500 | optimista: -11029


---
## Paso 7 · La matriz de decisión

Una **matriz de decisión** es una tabla con las opciones en las filas, los escenarios en las columnas y el resultado de cada combinación en cada celda. La opción A3 (todos) es la suma de la campaña a inactivos más la campaña a activos.

In [7]:
matriz = pd.DataFrame({
    "Pesimista": [0, beneficio_inactivos_pesimista, beneficio_inactivos_pesimista + beneficio_activos_pesimista],
    "Puntual": [0, beneficio_inactivos_puntual, beneficio_inactivos_puntual + beneficio_activos_puntual],
    "Optimista": [0, beneficio_inactivos_optimista, beneficio_inactivos_optimista + beneficio_activos_optimista],
}, index=["A1 · No hacer campaña", "A2 · Sólo inactivos", "A3 · Todos"])
print("Beneficio neto total en euros:")
print(matriz.round(0))

Beneficio neto total en euros:
                       Pesimista  Puntual  Optimista
A1 · No hacer campaña        0.0      0.0        0.0
A2 · Sólo inactivos      15910.0  38000.0    60090.0
A3 · Todos              -34061.0   7500.0    49061.0


---
## Paso 8 · Los tres criterios de decisión

Un **criterio de decisión** es una regla para elegir una opción cuando el resultado depende de un escenario que no se conoce. Se usan tres:

| Criterio | Pregunta que responde | Regla |
|---|---|---|
| Valor esperado | ¿Qué opción gana más si todo sale como se estimó? | Elegir el valor más alto de la columna "Puntual" |
| Maximin | ¿Qué opción deja el mejor resultado si todo sale mal? | Tomar el peor resultado de cada fila y elegir el más alto |
| Mínimo arrepentimiento | ¿Con qué opción se pierde menos frente a la mejor opción de cada escenario? | Calcular cuánto se deja de ganar en cada celda, tomar el máximo de cada fila y elegir el más bajo |

In [8]:
mejor_por_valor_esperado = matriz["Puntual"].idxmax()
peor_resultado_por_opcion = matriz.min(axis=1)
mejor_por_maximin = peor_resultado_por_opcion.idxmax()
print("Peor resultado de cada opción:")
print(peor_resultado_por_opcion.round(0))
print("Mejor opción por valor esperado:", mejor_por_valor_esperado)
print("Mejor opción por maximin:", mejor_por_maximin)

Peor resultado de cada opción:
A1 · No hacer campaña        0.0
A2 · Sólo inactivos      15910.0
A3 · Todos              -34061.0
dtype: float64
Mejor opción por valor esperado: A2 · Sólo inactivos
Mejor opción por maximin: A2 · Sólo inactivos


In [9]:
mejor_resultado_por_escenario = matriz.max(axis=0)
arrepentimiento = mejor_resultado_por_escenario - matriz
maximo_arrepentimiento = arrepentimiento.max(axis=1)
mejor_por_arrepentimiento = maximo_arrepentimiento.idxmin()
print("Arrepentimiento (lo que se deja de ganar frente a la mejor opción de cada escenario):")
print(arrepentimiento.round(0))
print("Máximo arrepentimiento de cada opción:")
print(maximo_arrepentimiento.round(0))
print("Mejor opción por mínimo arrepentimiento:", mejor_por_arrepentimiento)

Arrepentimiento (lo que se deja de ganar frente a la mejor opción de cada escenario):
                       Pesimista  Puntual  Optimista
A1 · No hacer campaña    15910.0  38000.0    60090.0
A2 · Sólo inactivos          0.0      0.0        0.0
A3 · Todos               49971.0  30500.0    11029.0
Máximo arrepentimiento de cada opción:
A1 · No hacer campaña    60090.0
A2 · Sólo inactivos          0.0
A3 · Todos               49971.0
dtype: float64
Mejor opción por mínimo arrepentimiento: A2 · Sólo inactivos


**Conclusión:** los tres criterios eligen **A2 · Sólo inactivos**. Cuando reglas distintas llevan a la misma opción, la decisión es más sólida.

---
## Paso 9 · Sensibilidad: ¿y si la campaña funciona peor?

La efectividad de 25 % es un **supuesto**, no un dato. Un **análisis de sensibilidad** cambia un supuesto para ver si la decisión cambia.

¿Cuál es la efectividad mínima para que la campaña a inactivos gane dinero incluso en el escenario pesimista? Se despeja de $B = 0$ con el límite inferior:

$$e_{\text{mínima}} = \frac{c}{p_{\text{inferior}} \cdot V}$$

In [10]:
efectividad_minima = costo_contacto / (inferior_inactivos * valor_cliente)
print("Efectividad mínima necesaria:", round(efectividad_minima * 100, 1), "%")
print("Efectividad supuesta:", efectividad * 100, "%")

Efectividad mínima necesaria: 22.0 %
Efectividad supuesta: 25.0 %


**Lectura:** si la campaña retiene a menos del 22 % de los clientes que se iban a ir, la decisión podría perder dinero. El supuesto (25 %) está por encima, pero la diferencia es pequeña. Por eso la recomendación incluye **medir la efectividad real con un piloto** antes de aplicar la campaña a todos los inactivos.

---
## Paso 10 · Cuánto dinero está en juego

El dinero que se llevan los clientes que abandonan se estima multiplicando el número de clientes que abandonan por el saldo promedio de quien abandona (intervalos P1 y P2):

In [11]:
dinero_inferior = N * 0.1822 * 83664.74
dinero_puntual = N * 0.206 * 91561.53
dinero_superior = N * 0.2298 * 99458.33
print("Dinero que se va (escenario bajo, millones de €):", round(dinero_inferior / 1e6, 1))
print("Dinero que se va (estimación puntual, millones de €):", round(dinero_puntual / 1e6, 1))
print("Dinero que se va (escenario alto, millones de €):", round(dinero_superior / 1e6, 1))

Dinero que se va (escenario bajo, millones de €): 152.4
Dinero que se va (estimación puntual, millones de €): 188.6
Dinero que se va (escenario alto, millones de €): 228.6


**Nota:** este rango combina los límites de dos intervalos. Es un rango aproximado, no un intervalo de confianza calculado con una fórmula.

---
## Paso 11 · Las consecuencias de equivocarse

| Error | Cuándo pasa | Consecuencia |
|---|---|---|
| Hacer la campaña cuando no convenía | La efectividad real es menor a 22 % | Se pierde parte del costo: como máximo unos 115,750 € (25 € × 4,630 inactivos). Es una pérdida limitada y se detecta pronto |
| No hacer la campaña cuando sí convenía | La campaña habría sido rentable | Se pierden 500 € al año por cada cliente que se pudo retener, y además sus depósitos. Un cliente que se va rara vez regresa |

El error de **no actuar** es más costoso y más difícil de revertir. Esto apoya la decisión de actuar.

---
## Paso 12 · La recomendación

1. Hacer la campaña de retención **sólo a clientes inactivos**.
2. No incluir a los clientes activos: su proporción de abandono está, con 95 % de confianza, por debajo del umbral.
3. Empezar con un **piloto**: 500 inactivos con campaña y 500 sin campaña, elegidos por sorteo. Comparar la proporción de abandono de los dos grupos con un intervalo para diferencia de proporciones (la técnica de P3) para medir la efectividad real.
4. No decir que los clientes se van **porque** son inactivos. Decir que la inactividad **identifica** a un grupo donde la campaña es rentable.

---
## Paso 13 · Decisiones metodológicas

El documento pide responder ocho preguntas **con sus propias palabras**. Abajo está el contenido de cada respuesta. Cada integrante debe reescribirlas con su forma de hablar antes de entregar.

**1. ¿Por qué seleccionaron esa población?**
Porque la campaña que queremos decidir aplica a toda la cartera de clientes del banco. El dataset contiene a los 10,000 clientes de Francia, Alemania y España, todos mayores de edad y con su abandono registrado.

**2. ¿Por qué utilizaron ese tamaño de muestra?**
Porque la fórmula del tamaño de muestra para una proporción, con margen de ±3 puntos, 95 % de confianza y p = 0.5, pide 965 clientes después de corregir por la población de 10,000. Redondeamos a 1,000.

**3. ¿Cómo seleccionaron a los participantes?**
Con muestreo aleatorio simple sin reemplazo: la computadora sorteó 1,000 clientes con la semilla 2026. Comprobamos con la prueba ji cuadrada que la muestra se parece a la población.

**4. ¿Por qué utilizaron esa técnica estadística?**
Porque cada pregunta tiene un tipo de variable distinto: proporción para el abandono (sí/no), media con t para el saldo, diferencia de proporciones para comparar inactivos y activos, y diferencia de medias de Welch para comparar edades sin suponer varianzas iguales.

**5. ¿Por qué utilizaron 95 % de confianza?**
Porque equilibra seguridad y precisión, es el estándar y los errores de esta decisión son económicos y corregibles. Con 99 % los intervalos serían 31 % más anchos.

**6. ¿Qué supuestos tuvieron que considerar?**
Muestra aleatoria, observaciones independientes, al menos 10 casos de cada resultado para las proporciones, normalidad de las medias por el Teorema del Límite Central (comprobada con bootstrap), y en la decisión, el costo, la efectividad y el valor del cliente.

**7. ¿Qué limitaciones tiene su estudio?**
No se conoce el banco ni las fechas; el estudio es observacional y no prueba causas; el intervalo no cubre errores de registro; los cuatro intervalos juntos tienen menos de 95 % de confianza (uno falló en la validación); y la efectividad de la campaña es supuesta.

**8. ¿Qué decisión podría tomar una organización con sus resultados?**
Hacer una campaña de retención dirigida a los clientes inactivos, empezando con un piloto para medir su efectividad real.

---
## Preguntas que puede hacer el profesor (apartado 7)

**1. ¿Qué decisión tomarían?**
Campaña de retención sólo a clientes inactivos, empezando con un piloto.

**2. ¿De dónde sale el umbral de 0.20?**
De igualar el beneficio a cero: costo entre efectividad por valor del cliente, 25 / (0.25 × 500) = 0.20.

**3. ¿Por qué usaron todo el intervalo y no sólo la estimación puntual?**
Porque la estimación puntual no considera la incertidumbre. Si todo el intervalo está arriba del umbral, la campaña conviene incluso en el peor caso razonable.

**4. ¿Por qué no hacer la campaña a todos?**
Porque en los activos todo el intervalo está debajo del umbral: contactarlos pierde dinero. En el escenario pesimista, A3 pierde 34,061 €.

**5. ¿Qué es el criterio maximin?**
Elegir la opción cuyo peor resultado sea el mejor. A2 tiene el mejor peor caso: 15,910 €.

**6. ¿De dónde salen los costos?**
Son supuestos ilustrativos; los datos no los incluyen. Por eso hicimos un análisis de sensibilidad.

**7. ¿Qué pasa si la campaña es menos efectiva?**
Si retiene a menos del 22 % de quienes se iban a ir, podría perder dinero. Por eso recomendamos un piloto.

**8. ¿Cuáles son las consecuencias de equivocarse?**
Actuar cuando no convenía cuesta como máximo unos 116 mil euros. No actuar cuando convenía significa perder clientes y sus depósitos, que difícilmente regresan.

---
## Tarea del Día 6 (para todo el equipo)

1. Cada integrante reescribe con sus palabras dos o tres de las ocho decisiones metodológicas (repártanlas).
2. Una persona explica en voz alta la matriz de decisión del Paso 7 y los tres criterios del Paso 8.

➡️ **Día 7:** ensayo de la exposición.